## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [1]:
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from math import ceil

In [2]:
effect_size = sms.proportion_effectsize(0.2, 0.19)
effect_size

np.float64(0.025241594409087353)

In [3]:
# так як в умові не вказано статистичну значущість, то припускаємо, що вона дорівнює 0.05
# потужність використовуємо ту, яку зазвичай використовують на практиці, тобто 0.8

required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
)
required_n = ceil(required_n)                          
print(f'Нам необхідно мінімум {required_n} спостережень для кожної групи')
print(f'Тобто в сумі для 2 груп потрібно {required_n*2} спостережень')

Нам необхідно мінімум 24638 спостережень для кожної групи
Тобто в сумі для 2 груп потрібно 49276 спостережень


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [4]:
df = pd.read_csv('statistical_hypothesis/cookie_cats.csv')
df.columns

Index(['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7'], dtype='str')

In [5]:
# перевіряємо чи є в даних про користувачів дублікати
users_counts = df['userid'].value_counts(ascending=False)
multi_users = users_counts[users_counts > 1].count()

print(f'Є {multi_users} користувачів, які зустрічаються кілька разів у наборі даних.')

Є 0 користувачів, які зустрічаються кілька разів у наборі даних.


In [6]:
mean_ret7 = df.groupby('version')['retention_7'].mean() 
mean_ret7

version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64

Гіпотеза: 

gate_30 дає краще утримання на 7-й день, ніж gate_40 (19,0% проти 18,2%). 
 
Це поки лише припущення на основі вибіркових середніх, його треба перевірити z-тестом.

H₀: утримання на 7-й день однакове в обох групах (p₃₀ = p₄₀)
H₁: утримання на 7-й день у групах відрізняється (p₃₀ ≠ p₄₀)


3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [7]:
# Вибірка у нас велика, тому можемо використовувати нормальне наближення для обчислення нашого значення  p  - тобто z-тест для пропорцій
# так як перевіряємо гіпотезу на нерівність показників утримання 2х варіантів воріт, застосовуємо двосторонній тест

gate30_results = df[df['version']=='gate_30']['retention_7']
control_cnt = gate30_results.count()

gate40_results = df[df['version']=='gate_40']['retention_7']
treatment_cnt = gate40_results.count()

In [8]:
successes = [gate30_results.sum(), gate40_results.sum()]
nobs = [control_cnt, treatment_cnt]

In [9]:
z_stat, pval = proportions_ztest(successes, nobs=nobs)
(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(successes, nobs=nobs, alpha=0.05)

print(f'z statistic: {z_stat:.2f}')
print(f'p-value: {pval:.4f}')
print(f'Довірчий інтервал 95% для групи control: [{lower_con:.4f}, {upper_con:.4f}]')
print(f'Довірчий інтервал 95% для групи treatment: [{lower_treat:.4f}, {upper_treat:.4f}]')


z statistic: 3.16
p-value: 0.0016
Довірчий інтервал 95% для групи control: [0.1866, 0.1938]
Довірчий інтервал 95% для групи treatment: [0.1785, 0.1855]


Висновок:

1. Так, різниця статистично значуща: p-value ≈ 0.0016 < 0.05, тому відхиляємо H₀ про рівність утримання на 7-й день. Утримання у групі gate_30 (19,0%) вище, ніж у gate_40 (18,2%), тобто переміщення воріт на рівень 40 знизило утримання приблизно на 0,8 в.п.

2. Ні, довірчі інтервали не перетинаються: [0.1866, 0.1938] для control і [0.1785, 0.1855] для treatment. Це означає, що навіть за найгіршої для control межі (0.1866) утримання вище за найкращу межу treatment (0.1855). Це узгоджується з результатом z-тесту.

Отже, рекомендуємо залишити ворота на рівні 30.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


  Гіпотеза: 

H₀: версія гри та утримання на 7-й день незалежні (зв'язку немає)

H₁: між версією гри та утриманням на 7-й день є залежність


In [10]:
crosstab = pd.crosstab(df['version'], df['retention_7'])

In [11]:
chi2, p, dof, expected = stats.chi2_contingency(crosstab)

print(f"p-value = {p:.5f}")


p-value = 0.00160


Висновок

p-value ≈ 0.0016 < 0.05, тому відхиляємо H₀ про незалежність версії гри та утримання на 7-й день, тобто залежність існує.
Проте, тест не показує в який бік відхиляється показник утримання залежно від версії гри (в якій версії утримання краще).